# ReefScan AI — Colab GPU training (E4 + E5 + E6)

Runs the **fish species** experiment matrix on fishnet.ai crops (feature-extraction screen → fine-tune the winner) + XAI/enhancement benchmarks.

**Before Run All:** Runtime → *Change runtime type* → **T4 GPU**. Duration ≈ 2.5–3.5 h. If *Allocating runtime* hangs, retry later — at the end the artifact zip downloads straight to your PC.

In [ ]:
# --- 0. Clone (or update) the repo + install dependencies ------------------
# Re-runs auto-update the code (git pull) while PRESERVING your trained
# artefacts inside the clone — never re-download, never lose checkpoints.
import os
repo = "/content/reefscan"
if not os.path.exists(repo):
    !git clone https://github.com/Jayant1711/reefscan.git {repo}
else:
    # fetch + reset: tracked files update to origin even if a previous phase
    # regenerated tracked files locally (the ledger) — untracked artifacts
    # (models/results/runs) are never touched.
    !git -C {repo} fetch origin
    !git -C {repo} reset --hard origin/main
%cd {repo}
!pip install -q -r requirements.txt

if os.path.basename(os.getcwd()) != os.path.basename(repo):
    os.chdir(repo)  # kernel restarts land elsewhere — re-enter the repo

In [ ]:
# --- 1. GPU + test-suite check (fails loudly if anything is broken) --------
import torch, subprocess, sys
assert torch.cuda.is_available(), "NO GPU — enable the accelerator and re-run!"
print("GPU:", torch.cuda.get_device_name(0))
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"])
assert r.returncode == 0, "TEST SUITE FAILED — fix before training (do not skip!)"


In [ ]:
# --- 2. FishNet v0.1.2 from fishnet.ai S3 (5.7 GB) -> bbox crops ----------
!python download_data.py --dataset fishnet --platform colab
!python download_data.py --prepare-fishnet --platform colab
import glob
n_crops = len(glob.glob("/content/reefscan_data/fishnet/curated/*/*/*.jpg"))
print("fish crops materialized:", n_crops)
assert n_crops > 1000, "too few crops - re-check the download and prepare cells above"


In [ ]:
# --- 3. Data cleaning (corrupt/dup/leakage scan + quarantine) --------------
!python clean_data.py --dataset fishnet --apply --platform colab

In [ ]:
# --- 4. E4 — fish feature-extraction screen (all 4 backbones, ~1.5 h) ------
!python train.py --task fish --model all --strategy feature_extraction --platform colab
import pandas as pd
pd.read_csv("results/fish/model_leaderboard.csv")

In [ ]:
# --- 5. E5 — auto-pick the winner and fine-tune it (~1.5 h) -----------------
import pandas as pd, subprocess, sys
lb = pd.read_csv("results/fish/model_leaderboard.csv")
fe = lb[lb["strategy"] == "feature_extraction"]
assert len(fe) >= 1, f"FE screen has no rows - E4 did not complete (only {len(lb)} rows total)"
if len(fe) < 4:
    print(f"WARNING: FE screen incomplete ({len(fe)}/4) - proceeding with available backbones")
winner = fe.nlargest(1, "weighted_f1")["model"].iloc[0]
print("Fine-tuning FE-screen winner:", winner)
r = subprocess.run([sys.executable, "train.py", "--task", "fish", "--model", winner,
                    "--strategy", "finetune", "--platform", "colab"])
if r.returncode != 0:
    print(f"WARNING: fine-tune failed for {winner} - evaluation below uses the FE checkpoint")
!python evaluate.py --task fish --platform colab
pd.read_csv("results/fish/model_leaderboard.csv")

In [ ]:
# --- 6. E6 — XAI grid + enhancement benchmark on real data ------------------
import glob, shlex
samples = (glob.glob("/content/reefscan_data/fishnet/curated/test/*/*.jpg") or
           glob.glob("/content/reefscan_data/fishnet/curated/train/*/*.jpg"))
!python evaluate.py --enhancement --data /content/reefscan_data/fishnet/curated/train
if samples:
    !python evaluate.py --xai --image {shlex.quote(str(samples[0]))}
else:
    print("No curated fish images found — skipping XAI (not fatal).")

In [ ]:
# --- 9. Save everything: ledger + artifact zip + BROWSER DOWNLOAD ----------
!python scripts/log_experiment.py
!python scripts/export_artifacts.py
import glob
from google.colab import files
zip_path = sorted(glob.glob("reefscan_artifacts_colab_*.zip"))[-1]
files.download(zip_path)   # saves the trained models to your PC
